# DistilBERT adaptation ablation with optional structured fusion

This notebook preserves two matched fine-tuning experiments on the same five grouped splits:

- **Top-two:** embeddings and transformer layers 0–3 stay exactly frozen; layers 4–5, the
  pre-classifier and final classifier are fine-tuned (about 14.77M trainable parameters). Two of
  the six layers is a fixed partial budget chosen in advance, not a tuned value.
- **Full:** all approximately 66.96M encoder and classification-head parameters are fine-tuned.

Each has a text-only arm and a structured late-fusion arm using the same two-logit classification-head
design. The four model variants are stored separately in `outputs/distilbert_top2` and
`outputs/finetuned`.

All arms use a pinned base checkpoint, maximum 512 tokens with dynamic batch padding, unweighted
cross-entropy (no class weighting, the same rule as the classical models), batch 16, AdamW, validation-AP checkpointing and validation-F1 thresholds.
There is no PCA, resampling, or hyperparameter search: the fixed recipe below lies inside the
fine-tuning ranges recommended for BERT (learning rate 2e-5 to 5e-5, batch 16 or 32, 2 to 4 epochs). One declared optimization seed is used
per data split, so split and optimization variability are not separately identified.

**Execution safety:** training requires CUDA and only runs with both `ON_GPU_SERVER = True` and
`RUN_TRAINING = True`. Completed model/seed pairs resume automatically. The notebook compares saved
results only after all 20 runs pass the final artifact checks. Notebook 04 contains the broader
comparison against the classical baselines.

In [1]:
import hashlib
import json
import os
import sys
import time
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

os.environ.setdefault("HF_HOME", str(Path("features/hf_cache").resolve()))
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import joblib
import numpy as np
import pandas as pd
import torch
import transformers
from safetensors import safe_open
from safetensors.torch import load_file, save_file
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from modules import data_helper as dh
from modules import eval_helper as ev
from modules import features as ft
from modules import report as rp
from modules import results_helper as rh

SEEDS = list(range(5))
ARMS = {"text": False, "fused": True}
EXPERIMENTS = {
    "top2": {
        "out": Path("outputs/distilbert_top2"), "group": "finetuned_top2",
        "trainable_layers": 2,
        "names": {"text": "distilbert_top2_text", "fused": "distilbert_top2_fused"},
        "metadata": {
            "training_strategy": "freeze embeddings and layers 0-3; train layers 4-5 plus head",
            "trainable_transformer_layers": [4, 5],
        },
    },
    "full": {
        "out": Path("outputs/finetuned"), "group": "finetuned",
        "trainable_layers": None,
        "names": {"text": "distilbert_text", "fused": "distilbert_fused"},
        "metadata": {
            "training_strategy": "full encoder and classification-head fine-tuning",
            "trainable_transformer_layers": None,
        },
    },
}
ACTIVE_EXPERIMENTS = ("top2", "full")
MODEL_NAME = "distilbert-base-uncased"
MODEL_REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"
MAX_LEN = 512
MICRO_BATCH = 16
EVAL_BATCH = 64
MAX_EPOCHS = 4
PATIENCE = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.1
DEVICE = "cuda:5"
ON_GPU_SERVER = True
PILOT_ONLY = False
RUN_TRAINING = False  # Set True together with ON_GPU_SERVER for intentional training/smoke tests.
VALIDATE_ON_GPU = True  # Set True only for optional inference-time checkpoint reload checks.
if not ON_GPU_SERVER or not RUN_TRAINING:
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("effective batch", MICRO_BATCH, "| maximum tokens", MAX_LEN)

torch 2.8.0+cu128 | transformers 4.57.6
effective batch 16 | maximum tokens 512


## 1. Data integrity and five saved splits

Every arm reloads the same saved indices used by notebook 02. Related-ad groups are evaluation
constraints, not model inputs. The checks below cover every row exactly once, pairwise group
separation, exact class supports, and the cleaned-data fingerprint.

In [2]:
dfe = ft.add_engineered_features(dh.load_processed())
y = dfe[dh.TARGET].to_numpy(dtype=int)
split_cache = {seed: dh.load_splits(f"splits/seed{seed}.npz", dfe) for seed in SEEDS}

for seed, splits in split_cache.items():
    ordered = np.concatenate([splits[name] for name in ("train", "val", "test")])
    assert np.array_equal(np.sort(ordered), np.arange(len(dfe)))
    for left_index, left in enumerate(("train", "val", "test")):
        for right in ("train", "val", "test")[left_index + 1:]:
            assert set(dfe.iloc[splits[left]]["group"]).isdisjoint(
                dfe.iloc[splits[right]]["group"]
            )
    print(
        f"seed {seed}:",
        {name: {"rows": len(splits[name]), "fraud": int(y[splits[name]].sum())}
         for name in ("train", "val", "test")},
    )

print("cleaned rows", len(dfe), "| related-ad groups", dfe["group"].nunique())

seed 0: {'train': {'rows': 10805, 'fraud': 535}, 'val': {'rows': 3447, 'fraud': 151}, 'test': {'rows': 3345, 'fraud': 170}}
seed 1: {'train': {'rows': 10314, 'fraud': 500}, 'val': {'rows': 3564, 'fraud': 175}, 'test': {'rows': 3719, 'fraud': 181}}
seed 2: {'train': {'rows': 10401, 'fraud': 537}, 'val': {'rows': 3758, 'fraud': 182}, 'test': {'rows': 3438, 'fraud': 137}}
seed 3: {'train': {'rows': 10614, 'fraud': 542}, 'val': {'rows': 3784, 'fraud': 162}, 'test': {'rows': 3199, 'fraud': 152}}
seed 4: {'train': {'rows': 10818, 'fraud': 529}, 'val': {'rows': 3190, 'fraud': 140}, 'test': {'rows': 3589, 'fraud': 187}}
cleaned rows 17597 | related-ad groups 4482


## 2. Fixed token policy and training-data coverage audit

The token ceiling is fixed a priori at DistilBERT's 512-position maximum; it is **not** selected from
validation or test AP. Dynamic batch padding avoids padding every batch to 512. The audit uses only
seed-0 training rows and documents lost text rather than tuning on it. Five text fields are joined in
the same order as notebook 02. Later fields can be lost when a document exceeds the ceiling.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
audit = dfe.iloc[split_cache[0]["train"]]
audit_texts = audit["text"].tolist()
encoded_audit = tokenizer(audit_texts, add_special_tokens=True, truncation=False, verbose=False)["input_ids"]
token_lengths = np.asarray([len(ids) for ids in encoded_audit])
print("seed-0 training token quantiles", {
    f"p{q}": float(np.percentile(token_lengths, q)) for q in (50, 90, 95, 99, 100)
})
for label in (0, 1):
    values = token_lengths[audit[dh.TARGET].to_numpy() == label]
    print(
        "class", label,
        "rows", len(values),
        "p50/p95/max", [float(np.percentile(values, q)) for q in (50, 95, 100)],
        "fully covered at 512", round(float(np.mean(values <= MAX_LEN)), 4),
        "mean removed-token fraction", round(
            float(np.mean(np.maximum(values - MAX_LEN, 0) / values)), 4
        ),
    )

# Approximate sequential per-field survival using separately tokenized fields and a two-token overhead.
field_lengths = []
for column in ft.TEXT_COLS:
    encoded = tokenizer(
        audit[column].fillna("").astype(str).tolist(), add_special_tokens=False, truncation=False,
        verbose=False
    )["input_ids"]
    field_lengths.append(np.asarray([len(ids) for ids in encoded]))
field_lengths = np.column_stack(field_lengths)
cumulative_before = np.column_stack([
    np.zeros(len(audit), dtype=int), np.cumsum(field_lengths[:, :-1], axis=1)
])
covered = np.clip(MAX_LEN - 2 - cumulative_before, 0, field_lengths)
nonempty = field_lengths > 0
survival = np.divide(
    covered, field_lengths, out=np.ones_like(covered, dtype=float), where=nonempty
)
print("mean sequential field-token survival at 512 among nonempty fields:")
for label in (0, 1):
    label_mask = audit[dh.TARGET].to_numpy() == label
    print("class", label, {
        column: round(float(survival[label_mask & nonempty[:, i], i].mean()), 4)
        for i, column in enumerate(ft.TEXT_COLS)
    })
del encoded_audit

seed-0 training token quantiles {'p50': 493.0, 'p90': 896.6000000000004, 'p95': 1053.0, 'p99': 1455.9599999999991, 'p100': 4076.0}
class 0 rows 10270 p50/p95/max [499.0, 1057.0, 4076.0] fully covered at 512 0.5211 mean removed-token fraction 0.1337
class 1 rows 535 p50/p95/max [337.0, 973.1999999999998, 1776.0] fully covered at 512 0.7159 mean removed-token fraction 0.0853


mean sequential field-token survival at 512 among nonempty fields:
class 0 {'title': 1.0, 'company_profile': 0.9973, 'description': 0.9171, 'requirements': 0.694, 'benefits': 0.5182}
class 1 {'title': 1.0, 'company_profile': 1.0, 'description': 0.9651, 'requirements': 0.8138, 'benefits': 0.7409}


## 3. Batch construction, fold-local structured features, and model

Text-only and fused arms share the same encoder and pre-classifier path. The fused arm changes only
the final classifier input. Nominal one-hot vocabularies, missing-value handling, and numeric scaling
are fitted on training rows **within each outer split**, never on validation or test rows.

In [4]:
class AdDataset(Dataset):
    def __init__(self, texts, labels, structured=None):
        self.encodings = tokenizer(
            list(texts), add_special_tokens=True, truncation=True, padding=False, max_length=MAX_LEN
        )
        self.labels = np.asarray(labels, dtype=np.int64)
        self.structured = structured
        assert len(self.encodings["input_ids"]) == len(self.labels)
        if structured is not None:
            assert len(structured) == len(self.labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.encodings.items()}
        if self.structured is not None:
            item["structured"] = torch.as_tensor(self.structured[index], dtype=torch.float32)
        item["labels"] = int(self.labels[index])
        return item


def make_loader(dataset, batch_size, shuffle=False, seed=0):
    generator = torch.Generator().manual_seed(seed)

    def collate(items):
        labels = torch.tensor([item.pop("labels") for item in items], dtype=torch.long)
        structured = None
        if "structured" in items[0]:
            structured = torch.stack([item.pop("structured") for item in items])
        batch = tokenizer.pad(items, padding=True, return_tensors="pt")
        batch["labels"] = labels
        if structured is not None:
            batch["structured"] = structured
        return batch

    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, generator=generator,
        num_workers=0, pin_memory=torch.cuda.is_available(), collate_fn=collate,
    )


def make_structured_preprocessor():
    nominal = Pipeline([
        ("missing", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    numeric = Pipeline([
        ("missing", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    binary = Pipeline([("missing", SimpleImputer(strategy="most_frequent"))])
    return ColumnTransformer([
        ("nominal", nominal, ft.NOMINAL_COLS),
        ("numeric", numeric, ft.NUMERIC_COLS),
        ("binary", binary, ft.BINARY_COLS),
    ], sparse_threshold=0.0)


class DistilBertFraudClassifier(nn.Module):
    """DistilBERT with a configurable encoder budget and shared text/fused head."""

    def __init__(self, structured_dim=0, trainable_encoder_layers=None):
        super().__init__()
        self.structured_dim = int(structured_dim)
        self.encoder = AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
        if trainable_encoder_layers is not None:
            first_trainable = len(self.encoder.transformer.layer) - trainable_encoder_layers
            assert 0 <= first_trainable < len(self.encoder.transformer.layer)
            for parameter in self.encoder.embeddings.parameters():
                parameter.requires_grad_(False)
            for index, layer in enumerate(self.encoder.transformer.layer):
                for parameter in layer.parameters():
                    parameter.requires_grad_(index >= first_trainable)
        config = self.encoder.config
        self.pre_classifier = nn.Linear(config.dim, config.dim)
        self.dropout = nn.Dropout(config.seq_classif_dropout)
        self.classifier = nn.Linear(config.dim + self.structured_dim, 2)

    def forward(self, input_ids, attention_mask, structured=None):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pooled = hidden[:, 0]
        pooled = self.dropout(nn.functional.relu(self.pre_classifier(pooled)))
        if self.structured_dim:
            assert structured is not None and structured.shape[1] == self.structured_dim
            pooled = torch.cat([pooled, structured.to(dtype=pooled.dtype)], dim=1)
        return self.classifier(pooled)


def move_batch(batch, device):
    return {key: value.to(device, non_blocking=True) for key, value in batch.items()}

## 4. Training and prediction helpers

Only parameters with `requires_grad=True` enter AdamW. Top-two mode freezes the embeddings and bottom
four encoder layers; full mode leaves every parameter trainable. The fixed recipe is AdamW
(LR `2e-5`, weight decay `0.01`), batch 16, 10% linear warmup/decay, gradient clipping,
bf16 autocast, at most four epochs, and early stopping after two non-improving validation-AP epochs.
The score for AP and thresholding is the fraud logit margin `z_fraud - z_real`.

In [5]:
def predict_scores(model, loader, device=DEVICE):
    model.eval()
    scores = []
    with torch.inference_mode():
        for batch in loader:
            batch = move_batch(batch, device)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device.startswith("cuda")):
                logits = model(
                    input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                    structured=batch.get("structured"),
                )
            margin = logits[:, 1].float() - logits[:, 0].float()
            scores.append(margin.cpu().numpy())
    return np.concatenate(scores)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, device=DEVICE):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    for batch_index, batch in enumerate(loader):
        batch = move_batch(batch, device)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device.startswith("cuda")):
            logits = model(
                input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                structured=batch.get("structured"),
            )
            loss = criterion(logits.float(), batch["labels"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        total_loss += float(loss.detach()) * len(batch["labels"])
        if batch_index and batch_index % 200 == 0:
            print(f"    step {batch_index}/{len(loader)} | loss {float(loss.detach()):.4f}", flush=True)
    return total_loss / len(loader.dataset)


def fine_tune(model, train_loader, val_loader, y_val, seed, device=DEVICE):
    dh.seed_everything(seed)
    model.to(device)
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    optimized_ids = {id(p) for group in optimizer.param_groups for p in group["params"]}
    assert optimized_ids == {id(p) for p in model.parameters() if p.requires_grad}
    total_steps = len(train_loader) * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=max(1, int(total_steps * WARMUP_FRACTION)),
        num_training_steps=total_steps,
    )
    criterion = nn.CrossEntropyLoss()  # Unweighted, like the classical models.

    history, best_state, best_ap, best_epoch, stale = [], None, -np.inf, 0, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        started = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, device)
        val_ap = float(ev.evaluate(y_val, predict_scores(model, val_loader, device), 0.0)["pr_auc"])
        epoch_seconds = time.time() - started
        history.append({
            "epoch": epoch, "train_loss": float(train_loss), "val_pr_auc": val_ap,
            "learning_rate": float(scheduler.get_last_lr()[0]),
            "completed_optimizer_steps": epoch * len(train_loader), "seconds": epoch_seconds,
        })
        print(
            f"    epoch {epoch}/{MAX_EPOCHS}: loss {train_loss:.4f} | val AP {val_ap:.4f} "
            f"| {epoch_seconds:.1f}s", flush=True,
        )
        if val_ap > best_ap:
            best_ap, best_epoch, stale = val_ap, epoch, 0
            best_state = {
                key: value.detach().to("cpu", copy=True)
                for key, value in model.state_dict().items()
            }
        else:
            stale += 1
            if stale >= PATIENCE:
                break

    assert best_state is not None
    model.load_state_dict(best_state)
    model.eval()
    assert all(torch.isfinite(value).all() for value in model.state_dict().values())
    return model, history, best_epoch

## 5. Correctness and memory smoke tests for both adaptation budgets

A balanced 16-ad batch must produce finite non-zero gradients and updates in every trainable
representative tensor. In top-two mode, embeddings and the first transformer layer must have exactly
zero gradients and zero weight change; in full mode, those tensors must change. The same batch is
briefly overfit as a correctness fixture, not a model-quality estimate. A separate maximum-length
batch measures the target micro-batch's peak GPU allocation.

In [6]:
def smoke_test(strategy):
    if not (ON_GPU_SERVER and RUN_TRAINING):
        print("GPU smoke test disabled")
        return
    assert torch.cuda.is_available()
    torch.cuda.set_device(DEVICE)
    torch.cuda.empty_cache()
    dh.seed_everything(0)
    train = dfe.iloc[split_cache[0]["train"]]
    real = train[train[dh.TARGET] == 0].iloc[:8]
    fraud = train[train[dh.TARGET] == 1].iloc[:8]
    smoke = pd.concat([real, fraud]).sample(frac=1, random_state=0)
    processor = make_structured_preprocessor().fit(train)
    structured = np.asarray(processor.transform(smoke), dtype=np.float32)
    dataset = AdDataset(smoke["text"], smoke[dh.TARGET], structured)
    loader = make_loader(dataset, MICRO_BATCH, shuffle=False)
    layers = EXPERIMENTS[strategy]["trainable_layers"]
    model = DistilBertFraudClassifier(structured.shape[1], layers).to(DEVICE)
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad], lr=1e-4
    )
    criterion = nn.CrossEntropyLoss()
    batch = move_batch(next(iter(loader)), DEVICE)

    frozen = {} if layers is None else {
        "word_embeddings": model.encoder.embeddings.word_embeddings.weight,
        "first_layer": model.encoder.transformer.layer[0].attention.q_lin.weight,
    }
    trainable_tensors = {
        "penultimate_layer": model.encoder.transformer.layer[-2].attention.q_lin.weight,
        "last_layer": model.encoder.transformer.layer[-1].attention.q_lin.weight,
        "pre_classifier": model.pre_classifier.weight,
        "classifier": model.classifier.weight,
    }
    if layers is None:
        trainable_tensors.update({
            "word_embeddings": model.encoder.embeddings.word_embeddings.weight,
            "first_layer": model.encoder.transformer.layer[0].attention.q_lin.weight,
        })
    representatives = {**frozen, **trainable_tensors}
    before = {name: value.detach().clone() for name, value in representatives.items()}
    losses = []
    model.train()
    for step in range(12):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(batch["input_ids"], batch["attention_mask"], batch["structured"])
            loss = criterion(logits.float(), batch["labels"])
        loss.backward()
        assert all(value.grad is not None and torch.isfinite(value.grad).all()
                   and torch.linalg.vector_norm(value.grad) > 0
                   for value in trainable_tensors.values())
        assert all(value.grad is None for value in frozen.values())
        optimizer.step()
        losses.append(float(loss.detach()))

    deltas = {name: float(torch.linalg.vector_norm(value.detach() - before[name]))
              for name, value in representatives.items()}
    assert all(delta > 0 for name, delta in deltas.items() if name in trainable_tensors), deltas
    assert all(deltas[name] == 0 for name in frozen), deltas
    assert losses[-1] < losses[0], losses
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    if layers is None:
        assert total == trainable
    else:
        assert 0 < trainable < total
    print(strategy, "trainable/total parameters", trainable, total)
    print(strategy, "representative update norms", deltas)
    print(strategy, "tiny-batch losses", [round(loss, 4) for loss in losses])

    optimizer.zero_grad(set_to_none=True)
    model.zero_grad(set_to_none=True)
    torch.cuda.synchronize(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)
    dummy_ids = torch.full((MICRO_BATCH, MAX_LEN), tokenizer.sep_token_id, device=DEVICE)
    dummy_mask = torch.ones_like(dummy_ids)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        logits = model(dummy_ids, dummy_mask, batch["structured"])
        memory_loss = criterion(logits.float(), batch["labels"])
    memory_loss.backward()
    torch.cuda.synchronize(DEVICE)
    print(strategy, "peak smoke allocation GiB",
          round(torch.cuda.max_memory_allocated(DEVICE) / 2**30, 2))
    assert torch.isfinite(memory_loss)
    del model, optimizer, batch, dummy_ids, dummy_mask
    torch.cuda.empty_cache()


for experiment_strategy in ACTIVE_EXPERIMENTS:
    smoke_test(experiment_strategy)

GPU smoke test disabled
GPU smoke test disabled


## 6. Fit, checkpoint, and save each split

Each run starts from the pinned base encoder, never from another split's learned weights. Validation
AP selects the best epoch. Validation predictions from that checkpoint select the F1 threshold; only
then are test predictions generated. Each completed run saves raw scores, metrics, per-epoch history,
model weights, tokenizer, structured preprocessing and a checkpoint checksum. Existing JSON/NPZ pairs
make completed runs resumable.

In [7]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        for block in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def run_experiment(strategy, arm, seed):
    experiment = EXPERIMENTS[strategy]
    out_dir = experiment["out"]
    model_name = experiment["names"][arm]
    use_structured = ARMS[arm]
    if rh.result_exists(out_dir, model_name, seed, fingerprint=dh.data_fingerprint(dfe),
                        val_indices=split_cache[seed]["val"],
                        test_indices=split_cache[seed]["test"]):
        print(f"seed {seed} {model_name}: already done, skipping", flush=True)
        return
    started = time.time()
    dh.seed_everything(seed)
    splits = split_cache[seed]
    train, val, test = (dfe.iloc[splits[name]] for name in ("train", "val", "test"))
    y_train, y_val, y_test = (part[dh.TARGET].to_numpy(dtype=int) for part in (train, val, test))

    processor = None
    train_structured = val_structured = test_structured = None
    if use_structured:
        processor = make_structured_preprocessor().fit(train)
        train_structured, val_structured, test_structured = (
            np.asarray(processor.transform(part), dtype=np.float32) for part in (train, val, test)
        )
        for values in (train_structured, val_structured, test_structured):
            assert np.isfinite(values).all()
        structured_dim = train_structured.shape[1]
        feature_names = processor.get_feature_names_out().tolist()
    else:
        structured_dim = 0
        feature_names = []

    train_dataset = AdDataset(train["text"], y_train, train_structured)
    val_dataset = AdDataset(val["text"], y_val, val_structured)
    test_dataset = AdDataset(test["text"], y_test, test_structured)
    train_loader = make_loader(train_dataset, MICRO_BATCH, shuffle=True, seed=seed)
    train_eval_loader = make_loader(train_dataset, EVAL_BATCH, shuffle=False)
    val_loader = make_loader(val_dataset, EVAL_BATCH, shuffle=False)
    test_loader = make_loader(test_dataset, EVAL_BATCH, shuffle=False)

    # dh.seed_everything above fixes fresh head initialization before model construction.
    model = DistilBertFraudClassifier(structured_dim, experiment["trainable_layers"])
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    if experiment["trainable_layers"] is None:
        assert total == trainable
    else:
        assert 0 < trainable < total
    print(
        f"seed {seed} {model_name}: {trainable:,}/{total:,} trainable parameters; "
        f"structured dimension {structured_dim}", flush=True,
    )
    model, history, best_epoch = fine_tune(
        model, train_loader, val_loader, y_val, seed=seed, device=DEVICE
    )

    score_train = predict_scores(model, train_eval_loader)
    score_val = predict_scores(model, val_loader)
    score_test = predict_scores(model, test_loader)
    threshold = ev.pick_threshold(y_val, score_val)
    val_metrics = ev.evaluate(y_val, score_val, threshold)
    test_metrics = ev.evaluate(y_test, score_test, threshold)
    train_metrics = ev.evaluate(y_train, score_train, threshold)

    checkpoint = out_dir / "checkpoints" / f"{model_name}_seed{seed}"
    checkpoint.mkdir(parents=True, exist_ok=True)
    weights = checkpoint / "model.safetensors"
    save_file({key: value.detach().cpu().contiguous()
               for key, value in model.state_dict().items()}, str(weights))
    tokenizer.save_pretrained(checkpoint)
    if processor is not None:
        joblib.dump(processor, checkpoint / "structured_preprocessor.joblib")
    config = {
        "model_name": model_name, "seed": seed, "base_model": MODEL_NAME,
        "base_revision": MODEL_REVISION, "pooling": "first token (CLS)",
        "structured": use_structured, "structured_dim": structured_dim,
        "feature_names": feature_names, "text_columns": ft.TEXT_COLS,
        "label_mapping": {"0": "legitimate", "1": "fraudulent"},
        "score": "fraud logit minus legitimate logit", "max_len": MAX_LEN,
        "micro_batch": MICRO_BATCH, "effective_batch": MICRO_BATCH,
        "epochs_requested": MAX_EPOCHS, "best_epoch": best_epoch, "patience": PATIENCE,
        "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "warmup_fraction": WARMUP_FRACTION, "loss": "unweighted cross-entropy",
        "total_parameters": total, "trainable_parameters": trainable,
        "torch": torch.__version__, "transformers": transformers.__version__,
        "data_fingerprint": dh.data_fingerprint(dfe),
    }
    config.update(experiment["metadata"])
    (checkpoint / "run_config.json").write_text(json.dumps(config, indent=2))
    checkpoint_hash = sha256(weights)
    seconds = time.time() - started
    rh.save_result(
        out_dir, model_name, experiment["group"], seed, threshold, val_metrics, test_metrics,
        y_val, score_val, y_test, score_test,
        train_pr_auc=train_metrics["pr_auc"],
        best_params={
            "base_model": MODEL_NAME, "base_revision": MODEL_REVISION,
            "pooling": "CLS", "fusion": "late" if use_structured else "none",
            **({"trained_layers": "encoder_4_5_plus_head"} if strategy == "top2" else {}),
            "max_len": MAX_LEN, "learning_rate": LEARNING_RATE,
            "effective_batch": MICRO_BATCH, "epochs": MAX_EPOCHS,
            "best_epoch": best_epoch, "loss": "unweighted_cross_entropy",
            "structured_dim": structured_dim,
        },
        seconds=seconds,
        notes=(("top-two encoder layers plus head fine-tuning" if strategy == "top2"
                else "full encoder fine-tuning")
               + f"; selected epoch {best_epoch} on validation AP; "
               + f"checkpoint {checkpoint.as_posix()}; safetensors sha256 {checkpoint_hash}"),
        history=history,
        data_fingerprint=dh.data_fingerprint(dfe),
        val_indices=splits["val"], test_indices=splits["test"],
    )
    print(
        f"seed {seed} {model_name}: epoch {best_epoch} | val AP {val_metrics['pr_auc']:.4f} "
        f"| test AP {test_metrics['pr_auc']:.4f} | F1 {test_metrics['f1']:.4f} "
        f"| {seconds:.1f}s", flush=True,
    )
    del model, train_loader, train_eval_loader, val_loader, test_loader
    del train_dataset, val_dataset, test_dataset
    torch.cuda.empty_cache()


if ON_GPU_SERVER and RUN_TRAINING:
    selected_seeds = [0] if PILOT_ONLY else SEEDS
    for experiment_strategy in ACTIVE_EXPERIMENTS:
        for experiment_seed in selected_seeds:
            for experiment_arm in ARMS:
                run_experiment(experiment_strategy, experiment_arm, experiment_seed)
else:
    print("training skipped; set ON_GPU_SERVER = True on the GPU server")

training skipped; set ON_GPU_SERVER = True on the GPU server


## 7. Verify every artifact and reload seed-0 checkpoints for both experiments

All 20 result pairs must match exact split labels/indices, the data fingerprint, array lengths, finite
scores and independently recomputed metrics. Checkpoint metadata, checksum, tokenizer and optional
preprocessor must also verify. With `VALIDATE_ON_GPU = True`, each of the four seed-0 checkpoints
is instantiated with fresh inference and must reproduce its saved test scores within tolerance.

In [8]:
def validate_saved_artifacts():
    verified_seeds = [0] if PILOT_ONLY else SEEDS
    for strategy in ACTIVE_EXPERIMENTS:
        experiment = EXPERIMENTS[strategy]
        out_dir = experiment["out"]
        for seed in verified_seeds:
            splits = split_cache[seed]
            for arm in ARMS:
                model_name = experiment["names"][arm]
                with open(out_dir / f"{model_name}_seed{seed}.json", encoding="utf-8") as source:
                    record = json.load(source)
                scores = rh.load_scores(out_dir, model_name, seed)
                rh.assert_result_lineage(record, scores, dh.data_fingerprint(dfe),
                                         splits["val"], splits["test"])
                for split_name in ("val", "test"):
                    expected = dfe.iloc[splits[split_name]][dh.TARGET].to_numpy(dtype=int)
                    assert np.array_equal(scores[f"y_{split_name}"], expected)
                    assert len(scores[f"score_{split_name}"]) == len(splits[split_name])
                    assert np.isfinite(scores[f"score_{split_name}"]).all()
                    recomputed = ev.evaluate(expected, scores[f"score_{split_name}"], record["thr"])
                    assert recomputed == record[split_name], (model_name, seed, split_name)
                config = json.loads((out_dir / "checkpoints" / f"{model_name}_seed{seed}" /
                                     "run_config.json").read_text())
                if experiment["trainable_layers"] is None:
                    assert config["total_parameters"] == config["trainable_parameters"]
                    expected_layers = None
                else:
                    assert 0 < config["trainable_parameters"] < config["total_parameters"]
                    expected_layers = [4, 5]
                assert config.get("trainable_transformer_layers") == expected_layers
                assert config["base_revision"] == MODEL_REVISION
                checkpoint_dir = out_dir / "checkpoints" / f"{model_name}_seed{seed}"
                weights_path = checkpoint_dir / "model.safetensors"
                assert sha256(weights_path) in record["notes"]
                saved_tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
                probe = "lineage audit tokenizer probe"
                assert saved_tokenizer(probe)["input_ids"] == tokenizer(probe)["input_ids"]
                if ARMS[arm]:
                    processor = joblib.load(checkpoint_dir / "structured_preprocessor.joblib")
                    assert processor.get_feature_names_out().tolist() == config["feature_names"]
                with safe_open(weights_path, framework="pt", device="cpu") as weights:
                    assert "encoder.embeddings.word_embeddings.weight" in weights.keys()
                    assert "classifier.weight" in weights.keys()
                print("verified", model_name, "seed", seed,
                      "lineage, checkpoint checksum, tokenizer, optional preprocessor, JSON/NPZ")

    if not VALIDATE_ON_GPU:
        print("GPU checkpoint reload skipped; set VALIDATE_ON_GPU = True on the GPU server")
        return
    for strategy in ACTIVE_EXPERIMENTS:
        experiment = EXPERIMENTS[strategy]
        out_dir = experiment["out"]
        for arm, use_structured in ARMS.items():
            model_name = experiment["names"][arm]
            checkpoint = out_dir / "checkpoints" / f"{model_name}_seed0"
            config = json.loads((checkpoint / "run_config.json").read_text())
            model = DistilBertFraudClassifier(
                config["structured_dim"], experiment["trainable_layers"]
            )
            state = load_file(str(checkpoint / "model.safetensors"), device="cpu")
            model.load_state_dict(state, strict=True)
            model.to(DEVICE).eval()
            frame = dfe.iloc[split_cache[0]["test"]]
            structured = None
            if use_structured:
                processor = joblib.load(checkpoint / "structured_preprocessor.joblib")
                assert processor.get_feature_names_out().tolist() == config["feature_names"]
                structured = np.asarray(processor.transform(frame), dtype=np.float32)
            dataset = AdDataset(frame["text"], frame[dh.TARGET], structured)
            loader = make_loader(dataset, EVAL_BATCH, shuffle=False)
            reloaded_scores = predict_scores(model, loader)
            expected_scores = rh.load_scores(out_dir, model_name, 0)["score_test"]
            assert np.allclose(reloaded_scores, expected_scores, rtol=1e-5, atol=1e-5), (
                model_name, float(np.max(np.abs(reloaded_scores - expected_scores)))
            )
            print("reloaded", model_name, "seed 0; max score difference",
                  float(np.max(np.abs(reloaded_scores - expected_scores))))
            del model, dataset, loader
            torch.cuda.empty_cache()


validate_saved_artifacts()

verified distilbert_top2_text seed 0 lineage, checkpoint checksum, tokenizer, optional preprocessor, JSON/NPZ


verified distilbert_top2_fused seed 0 lineage, checkpoint checksum, tokenizer, optional preprocessor, JSON/NPZ


verified distilbert_top2_text seed 1 lineage, checkpoint checksum, tokenizer, optional preprocessor, JSON/NPZ


verified distilbert_top2_fused seed 1 lineage, checkpoint checksum, tokenizer, optional preprocessor, JSON/NPZ


verified distilbert_top2_text seed 2 lineage, checkpoint checksum, tokenizer, optional preprocessor, JSON/NPZ


verified distilbert_top2_fused seed 2 lineage, checkpoint checksum, tokenizer, optional preprocessor, JSON/NPZ


... (15 outputs left out)

reloaded distilbert_top2_text seed 0; max score difference 0.0


reloaded distilbert_top2_fused seed 0; max score difference 0.0


reloaded distilbert_text seed 0; max score difference 0.0


reloaded distilbert_fused seed 0; max score difference 0.0


In [9]:
verified_seeds = [0] if PILOT_ONLY else SEEDS
frames = []
for strategy in ACTIVE_EXPERIMENTS:
    experiment = EXPERIMENTS[strategy]
    current = rh.load_results(experiment["out"])
    current = current[current["model"].isin(experiment["names"].values())
                      & current["seed"].isin(verified_seeds)].copy()
    current["strategy"] = strategy
    current["arm"] = current["model"].map({name: arm for arm, name in experiment["names"].items()})
    frames.append(current)
records = pd.concat(frames, ignore_index=True)
assert len(records) == 4 * len(verified_seeds)

columns = ["strategy", "arm", "seed", "val_pr_auc", "test_pr_auc",
           "test_precision", "test_recall", "test_f1", "seconds"]
display(records[columns].round(4).set_index(["strategy", "arm", "seed"]))

summary = records.groupby(["strategy", "arm"])[
    ["val_pr_auc", "test_pr_auc", "test_f1", "seconds"]
].agg(["mean", "std"]).round(4)
display(summary)

adaptation_rows = []
for arm in ARMS:
    full_name = EXPERIMENTS["full"]["names"][arm]
    top2_name = EXPERIMENTS["top2"]["names"][arm]
    full = records[records["model"] == full_name].set_index("seed")
    top2 = records[records["model"] == top2_name].set_index("seed")
    for metric, label in (("pr_auc", "AP"), ("f1", "F1")):
        for split in ("val", "test"):
            delta = full[f"{split}_{metric}"] - top2[f"{split}_{metric}"]
            adaptation_rows.append({
                "arm": arm, "metric": f"{split} {label} (full - top2)",
                **{f"s{seed}": delta.loc[seed] for seed in verified_seeds},
                "mean": delta.mean(), "positive": f"{int((delta > 0).sum())}/{len(delta)}",
            })
adaptation = pd.DataFrame(adaptation_rows).set_index(["arm", "metric"])
display(adaptation.round(4))
print("verified", len(records), "fine-tuning result pairs across both adaptation budgets")

val_pr_auc  test_pr_auc  test_precision  test_recall  \
strategy arm   seed                                                         
top2     fused 0         0.2972       0.5445          0.8721       0.4412   
               1         0.5972       0.5805          0.5355       0.5414   
               2         0.6712       0.3283          0.2266       0.3358   
               3         0.4843       0.6934          0.8000       0.5526   
               4         0.7688       0.6520          0.8909       0.2620   
         text  0         0.2782       0.5080          0.3586       0.5000   
               1         0.5815       0.6323          0.6696       0.4254   
               2         0.6887       0.2636          0.2091       0.3358   
               3         0.4571       0.6887          0.7387       0.5395   
               4         0.7165       0.4542          0.8974       0.1872   
full     fused 0         0.6368       0.5871          0.8667       0.4588   
               1         0.5553       0.6840          0.8014       0.6243   
               2         0.6503       0.3408          0.4333       0.2847   
               3         0.5749       0.7640          0.8817       0.5395   
               4         0.8674       0.6093          0.8515       0.4599   
         text  0         0.6705       0.6404          0.8625       0.4059   
               1         0.5366       0.6585          0.2007       0.6464   
               2         0.7243       0.4226          0.7407       0.2920   
               3         0.6239       0.6888          0.8140       0.4605   
               4         0.8375       0.6662          0.8519       0.6150   

                     test_f1   seconds  
strategy arm   seed                     
top2     fused 0      0.5859  110.2377  
               1      0.5385  126.3855  
               2      0.2706  125.4069  
               3      0.6537  110.4029  
               4      0.4050  117.4312  
         text  0      0.4177  122.4864  
               1      0.5203  125.5489  
               2      0.2577  129.5238  
               3      0.6236  134.4703  
               4      0.3097  128.3327  
full     fused 0      0.6000  140.1266  
               1      0.7019  164.6361  
               2      0.3436  134.5130  
               3      0.6694  167.2401  
               4      0.5972  165.4783  
         text  0      0.5520  156.7355  
               1      0.3063  157.7619  
               2      0.4188  161.6556  
               3      0.5882  164.8832  
               4      0.7143  169.5522

val_pr_auc         test_pr_auc         test_f1          \
                     mean     std        mean     std    mean     std   
strategy arm                                                            
full     fused     0.6570  0.1243      0.5971  0.1592  0.5824  0.1409   
         text      0.6786  0.1124      0.6153  0.1091  0.5159  0.1575   
top2     fused     0.5638  0.1817      0.5597  0.1420  0.4907  0.1531   
         text      0.5444  0.1805      0.5094  0.1664  0.4258  0.1499   

                 seconds           
                    mean      std  
strategy arm                       
full     fused  154.3988  15.7448  
         text   162.1177   5.2696  
top2     fused  117.9728   7.8017  
         text   128.0724   4.4911

s0      s1      s2      s3      s4    mean  \
arm   metric                                                                  
text  val AP (full - top2)   0.3923 -0.0449  0.0355  0.1668  0.1210  0.1341   
      test AP (full - top2)  0.1324  0.0262  0.1590  0.0001  0.2119  0.1059   
      val F1 (full - top2)   0.4439 -0.0403  0.0035  0.0907  0.1313  0.1258   
      test F1 (full - top2)  0.1343 -0.2140  0.1611 -0.0353  0.4046  0.0901   
fused val AP (full - top2)   0.3395 -0.0419 -0.0209  0.0906  0.0986  0.0932   
      test AP (full - top2)  0.0426  0.1035  0.0126  0.0706 -0.0427  0.0373   
      val F1 (full - top2)   0.3057 -0.0436  0.0045  0.0246  0.1463  0.0875   
      test F1 (full - top2)  0.0141  0.1634  0.0730  0.0157  0.1923  0.0917   

                            positive  
arm   metric                          
text  val AP (full - top2)       4/5  
      test AP (full - top2)      5/5  
      val F1 (full - top2)       4/5  
      test F1 (full - top2)      3/5  
fused val AP (full - top2)       3/5  
      test AP (full - top2)      4/5  
      val F1 (full - top2)       4/5  
      test F1 (full - top2)      5/5

verified 20 fine-tuning result pairs across both adaptation budgets


## 8. Explaining the fine-tuned model

The explained model is the full fine-tuned **text-only** model on seed 0. It is chosen by design, not by
score: it is the largest adaptation budget, and without structured inputs every prediction comes from
the text alone, so text attributions are complete. Two complementary views are computed on the
seed-0 test partition:

1. **Field occlusion.** Each of the five text fields is blanked for every test ad, the ads are re-scored,
   and the drop in test AP measures how much ranking quality depends on that field.
2. **Integrated gradients** (Sundararajan et al., 2017) attribute each fraudulent test ad's fraud
   margin to its tokens, integrating gradients along a straight path from a baseline that keeps
   `[CLS]`/`[SEP]` and replaces every other token by `[PAD]`. WordPiece pieces are merged into words,
   and words are summed over the fraud ads. The completeness check compares each ad's summed
   attribution with its margin difference from the baseline.

In [10]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

EXPLAIN_MODEL, EXPLAIN_SEED, IG_STEPS = "distilbert_text", 0, 32
EXPLAIN_DIR = EXPERIMENTS["full"]["out"] / "explanations"


def joined_text(frame, blank=None):
    """Rebuild the model text exactly as add_engineered_features does, optionally blanking one field."""
    parts = [frame[col].fillna("").astype(str) if col != blank else pd.Series("", index=frame.index)
             for col in ft.TEXT_COLS]
    text = parts[0]
    for part in parts[1:]:
        text = text + " " + part
    return text


def load_checkpoint(strategy, arm, seed):
    experiment = EXPERIMENTS[strategy]
    checkpoint = experiment["out"] / "checkpoints" / f"{experiment['names'][arm]}_seed{seed}"
    config = json.loads((checkpoint / "run_config.json").read_text())
    model = DistilBertFraudClassifier(config["structured_dim"], experiment["trainable_layers"])
    model.load_state_dict(load_file(str(checkpoint / "model.safetensors"), device="cpu"), strict=True)
    return model.to(DEVICE).eval()


def fraud_margin_from_embeddings(model, embeddings, attention_mask):
    """Text-only classifier forward pass starting from word embeddings (dropout is off in eval mode)."""
    hidden = model.encoder(inputs_embeds=embeddings, attention_mask=attention_mask).last_hidden_state
    pooled = nn.functional.relu(model.pre_classifier(hidden[:, 0]))
    logits = model.classifier(pooled)
    return logits[:, 1] - logits[:, 0]


def integrated_gradients(model, text):
    """Integrated gradients of the fraud margin w.r.t. word embeddings; baseline keeps [CLS]/[SEP], pads the rest."""
    encoded = tokenizer(text, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    ids, mask = encoded["input_ids"], encoded["attention_mask"]
    embed = model.encoder.embeddings.word_embeddings
    baseline_ids = torch.full_like(ids, tokenizer.pad_token_id)
    baseline_ids[0, 0], baseline_ids[0, -1] = ids[0, 0], ids[0, -1]
    with torch.no_grad():
        x, x0 = embed(ids), embed(baseline_ids)
    alphas = (torch.arange(IG_STEPS, device=DEVICE, dtype=x.dtype) + 0.5) / IG_STEPS  # midpoint rule
    path = (x0 + alphas[:, None, None] * (x - x0)).requires_grad_(True)
    margins = fraud_margin_from_embeddings(model, path, mask.expand(IG_STEPS, -1))
    grads = torch.autograd.grad(margins.sum(), path)[0].mean(dim=0)
    attributions = ((x - x0)[0] * grads).sum(dim=-1)
    with torch.no_grad():
        gap = float(fraud_margin_from_embeddings(model, x, mask) - fraud_margin_from_embeddings(model, x0, mask))
    tokens = tokenizer.convert_ids_to_tokens(ids[0])
    return tokens, attributions.detach().cpu().numpy(), gap


def word_attributions(tokens, values):
    """Merge WordPiece continuations into words and drop special tokens."""
    words = []
    for token, value in zip(tokens, values):
        if token in tokenizer.all_special_tokens:
            continue
        if token.startswith("##") and words:
            words[-1][0] += token[2:]
            words[-1][1] += value
        else:
            words.append([token, value])
    return words

In [11]:
if ON_GPU_SERVER:
    EXPLAIN_DIR.mkdir(parents=True, exist_ok=True)
    model = load_checkpoint("full", "text", EXPLAIN_SEED)
    test = dfe.iloc[split_cache[EXPLAIN_SEED]["test"]]
    y_test = test[dh.TARGET].to_numpy(dtype=int)
    assert joined_text(test).equals(test["text"]), "rebuilt text must equal the model input"

    def scores_for(texts):
        return predict_scores(model, make_loader(AdDataset(texts, y_test), EVAL_BATCH))

    base_scores = scores_for(test["text"])
    saved_scores = rh.load_scores(EXPERIMENTS["full"]["out"], EXPLAIN_MODEL, EXPLAIN_SEED)["score_test"]
    assert np.allclose(base_scores, saved_scores, atol=1e-3), "reloaded model must reproduce saved scores"
    base_ap = ev.evaluate(y_test, base_scores, 0.0)["pr_auc"]

    # 1. Field occlusion: blank one text field for every test ad and re-score.
    occlusion_rows = []
    for field in ft.TEXT_COLS:
        scores = scores_for(joined_text(test, blank=field))
        occlusion_rows.append({
            "field": field,
            "share of test ads with field": float(test[field].fillna("").str.strip().ne("").mean()),
            "test AP without field": ev.evaluate(y_test, scores, 0.0)["pr_auc"],
            "AP drop": base_ap - ev.evaluate(y_test, scores, 0.0)["pr_auc"],
            "mean margin change, fraud ads": float((scores - base_scores)[y_test == 1].mean()),
        })
    occlusion = pd.DataFrame(occlusion_rows)
    occlusion.to_csv(EXPLAIN_DIR / f"field_occlusion_seed{EXPLAIN_SEED}.csv", index=False)
    print(f"full test AP {base_ap:.4f}")
    display(occlusion.round(4))
    rp.write_table_rows("tab_field_occlusion.tex", [
        (rp.tex(row.field.replace("_", " ")), f"{100 * row._2:.1f}\\%", f"{row._3:.3f}", f"{row._4:+.3f}")
        for row in occlusion.itertuples()
    ])

    # 2. Integrated gradients for every fraudulent test ad, merged to words.
    word_rows, completeness = [], []
    for ad, text in enumerate(test.loc[y_test == 1, "text"]):
        tokens, values, gap = integrated_gradients(model, text)
        completeness.append(abs(values.sum() - gap) / max(abs(gap), 1e-6))
        word_rows += [{"ad": ad, "word": word, "attribution": float(value)}
                      for word, value in word_attributions(tokens, values)]
    words = pd.DataFrame(word_rows)
    # Same stop-word list as the TF-IDF features; punctuation and numbers are dropped too.
    words = words[words["word"].str.contains("[a-z]", regex=True) & (words["word"].str.len() > 1)
                  & ~words["word"].isin(ENGLISH_STOP_WORDS)]
    per_word = words.groupby("word").agg(ads=("ad", "nunique"), total=("attribution", "sum"))
    per_word["mean per fraud ad"] = per_word["total"] / int(y_test.sum())
    per_word = per_word[per_word["ads"] >= 5]
    top_words = pd.concat([
        per_word.nlargest(15, "total").assign(direction="towards fraud"),
        per_word.nsmallest(15, "total").assign(direction="towards real"),
    ]).reset_index()
    top_words.to_csv(EXPLAIN_DIR / f"ig_words_seed{EXPLAIN_SEED}.csv", index=False)
    print(f"integrated gradients on {int(y_test.sum())} fraud ads, {IG_STEPS} steps; "
          f"completeness error median {np.median(completeness):.3f}, 90th pct {np.percentile(completeness, 90):.3f}")
    display(top_words.round(4))
    fraud_side = top_words[top_words["direction"] == "towards fraud"].head(10)
    real_side = top_words[top_words["direction"] == "towards real"].head(10)
    rp.write_table_rows("tab_ig_words.tex", [
        (rp.tex(f.word), f"{f._4:+.3f}", f.ads, rp.tex(r.word), f"{r._4:+.3f}", r.ads)
        for f, r in zip(fraud_side.itertuples(), real_side.itertuples())
    ])
    del model
    torch.cuda.empty_cache()
else:
    print("explanations need the GPU; set ON_GPU_SERVER = True")

full test AP 0.6404


,field,share of test ads with field,test AP without field,AP drop,"mean margin change, fraud ads"
0,title,1.0000,0.6091,0.0314,0.0992
1,company_profile,0.7859,0.5434,0.0971,0.3212
2,description,1.0000,0.6344,0.0061,1.2104
3,requirements,0.8972,0.6403,0.0002,0.2723
4,benefits,0.7163,0.6537,-0.0133,0.3049


saved report table tab_field_occlusion.tex


integrated gradients on 170 fraud ads, 32 steps; completeness error median 0.003, 90th pct 0.013


,word,ads,total,mean per fraud ad,direction
0,available,57,31.6893,0.1864,towards fraud
1,assistant,21,23.5622,0.1386,towards fraud
2,positions,55,21.1272,0.1243,towards fraud
3,administrative,20,17.3197,0.1019,towards fraud
4,apply,41,15.5458,0.0914,towards fraud
5,seeking,54,15.1748,0.0893,towards fraud
6,clerical,17,14.7273,0.0866,towards fraud
7,entry,61,12.6658,0.0745,towards fraud
8,link,9,11.2974,0.0665,towards fraud
9,typist,20,9.8578,0.0580,towards fraud


saved report table tab_ig_words.tex


## Scope and next step

The ten paired full-versus-top-two comparisons are descriptive, not independent replications: every
outer split has one split-derived optimization seed, and the same data are reused across partitions.
Full-data EDA informed the features and earlier seed-0 test results were inspected, so all findings
remain exploratory. The 512-token policy discards class-unequal amounts of text, another limitation
that must accompany the analysis rather than be used for test-driven repairs.

Notebook 04 should compare all four fine-tuning arms with the classical baselines on identical
rows. It must preserve these adaptation and fusion pairings and must not pool overlapping test sets
as independent replications.